# Lab 1F-2: Modern Deep-Learning CLIP + FAISS

Exercise 1.F Pipeline 2 is the modern deep-learning pipeline: CLIP ViT-B/32 learns image/text embeddings, and FAISS retrieves vectors over the same Flickr8k data and the same `evaluation_queries.json` protocol used by Notebook 1.

SigLIP2 is a newer alternative vision-language encoder; this lab uses CLIP because it is simpler and stable for student execution.

Modern embeddings are expected to improve semantic matching, but results depend on query type and relevance definition.

## 0. Setup

Set `FAST_DEV=True` for a 200-image subset used during debugging. Set it to `False` for a full Flickr8k run and final saved outputs.

In [ ]:
FAST_DEV = True
FAST_DEV_LIMIT = 200

from pathlib import Path
import hashlib
import json
import re
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import CLIPModel, CLIPProcessor

try:
    import faiss
except ImportError as exc:
    raise ImportError('Install faiss-cpu before running this notebook.') from exc

LAB_DIR = Path('content/lab/1F')
if not (LAB_DIR / 'evaluation_queries.json').exists():
    LAB_DIR = Path('.')
OUTPUT_DIR = LAB_DIR / 'outputs'
FIGURE_DIR = OUTPUT_DIR / 'figures'
QUERY_FILE = LAB_DIR / 'evaluation_queries.json'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'FAST_DEV={FAST_DEV}, device={device}')

## 1. Load Flickr8k and Build Stable Image IDs

In [ ]:
from datasets import load_dataset, concatenate_datasets

print('Loading Flickr8k...')
ds = load_dataset('jxie/flickr8k')
full_ds = concatenate_datasets([ds['train'], ds['test']])
if FAST_DEV:
    full_ds = full_ds.select(range(min(FAST_DEV_LIMIT, len(full_ds))))
print(f'Images available in this run: {len(full_ds)}')


def normalize_text(text):
    return re.sub(r'[^a-z0-9]+', ' ', str(text).lower()).strip()


def caption_list(item):
    caps = item.get('caption', [])
    if isinstance(caps, str):
        caps = [caps]
    elif not caps:
        caps = [item.get(f'caption_{i}', '') for i in range(5)]
    cleaned = []
    for c in caps:
        if isinstance(c, dict):
            c = c.get('raw') or c.get('text') or ''
        s = str(c).strip()
        if s:
            cleaned.append(s)
    return cleaned


def stable_image_id(item, idx, image):
    for key in ('image_id', 'img_id', 'filename', 'file_name'):
        value = item.get(key)
        if value:
            return Path(str(value)).name
    filename = getattr(image, 'filename', '')
    if filename:
        return Path(filename).name
    captions = caption_list(item)
    first_caption = captions[0] if captions else ''
    w, h = image.size
    digest = hashlib.sha1(f'{first_caption}|{w}x{h}'.encode('utf-8')).hexdigest()[:12]
    return f'flickr8k_{digest}'


records = []
for idx in range(len(full_ds)):
    item = full_ds[idx]
    image = item['image'].convert('RGB')
    w, h = image.size
    captions = caption_list(item)
    records.append({
        'image_id': stable_image_id(item, idx, image),
        'dataset_pos': idx,
        'width': w,
        'height': h,
        'captions': captions,
        'all_captions': ' '.join(captions),
    })

df = pd.DataFrame(records)
dup_count = int(df['image_id'].duplicated().sum())
if dup_count:
    warnings.warn(f'Found {dup_count} duplicate image rows; keeping first occurrence per image_id.')
    df = df.drop_duplicates(subset='image_id', keep='first').reset_index(drop=True)
image_id_to_pos = dict(zip(df['image_id'], df['dataset_pos']))
ordered_image_ids = df['image_id'].tolist()

signature = {
    'fast_dev': FAST_DEV,
    'count': int(len(df)),
    'first_items': df[['image_id', 'width', 'height', 'captions']].head(5).to_dict(orient='records'),
}
signature_path = OUTPUT_DIR / ('dataset_signature_clip_fast_dev.json' if FAST_DEV else 'dataset_signature_clip_full.json')
if signature_path.exists():
    old_signature = json.loads(signature_path.read_text(encoding='utf-8'))
    if old_signature.get('first_items') != signature['first_items']:
        warnings.warn(f'Dataset signature changed: {signature_path}')
signature_path.write_text(json.dumps(signature, indent=2), encoding='utf-8')
df.head()

## 2. Load CLIP ViT-B/32

The implemented model is `openai/clip-vit-base-patch32`. The visual encoder divides images into fixed-size patches internally as part of representation learning; this is not the image-segmentation module from the Part E architecture.

In [ ]:
MODEL_NAME = 'openai/clip-vit-base-patch32'

print(f'Loading {MODEL_NAME}...')
model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model.eval()
EMBED_DIM = int(model.config.projection_dim)
print(f'Embedding dimension: {EMBED_DIM}')

## 3. Encode Images with Caching

Image embeddings are cached under `content/lab/1F/outputs/`. The cache key distinguishes FAST_DEV from the full run and stores the stable image-id order.

In [ ]:
cache_tag = 'fast_dev' if FAST_DEV else 'full'
embedding_path = OUTPUT_DIR / f'clip_image_embeddings_{cache_tag}.npy'
ids_path = OUTPUT_DIR / f'clip_image_ids_{cache_tag}.json'


def encode_image_batch(images):
    inputs = processor(images=images, return_tensors='pt', padding=True).to(device)
    with torch.no_grad():
        embeddings = model.get_image_features(**inputs)
    if not torch.is_tensor(embeddings):
        if hasattr(embeddings, 'image_embeds') and embeddings.image_embeds is not None:
            embeddings = embeddings.image_embeds
        elif hasattr(embeddings, 'pooler_output') and embeddings.pooler_output is not None:
            embeddings = embeddings.pooler_output
        elif hasattr(embeddings, 'last_hidden_state'):
            embeddings = embeddings.last_hidden_state[:, 0, :]
        else:
            raise TypeError(f'Unsupported CLIP embedding output type: {type(embeddings)}')
    embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return embeddings.cpu().numpy().astype('float32')


def encode_all_images(batch_size=32):
    batches = []
    for start in range(0, len(full_ds), batch_size):
        end = min(start + batch_size, len(full_ds))
        images = [full_ds[int(df.iloc[i]['dataset_pos'])]['image'].convert('RGB') for i in range(start, end)]
        batches.append(encode_image_batch(images))
        print(f'Encoded {end}/{len(full_ds)} images')
    print()
    return np.vstack(batches).astype('float32')


if embedding_path.exists() and ids_path.exists():
    cached_ids = json.loads(ids_path.read_text(encoding='utf-8'))
    if cached_ids == ordered_image_ids:
        image_embeddings = np.load(embedding_path).astype('float32')
        print(f'Loaded cached embeddings from {embedding_path}')
    else:
        warnings.warn('Cached image-id order changed; recomputing embeddings.')
        image_embeddings = encode_all_images()
        np.save(embedding_path, image_embeddings)
        ids_path.write_text(json.dumps(ordered_image_ids, indent=2), encoding='utf-8')
else:
    image_embeddings = encode_all_images()
    np.save(embedding_path, image_embeddings)
    ids_path.write_text(json.dumps(ordered_image_ids, indent=2), encoding='utf-8')

print(f'image_embeddings shape: {image_embeddings.shape}')

## 4. FAISS IndexFlatIP Main Result

For Flickr8k scale, `IndexFlatIP` is the main result because it gives exact inner-product retrieval over normalized embeddings.

In [ ]:
index_flat = faiss.IndexFlatIP(image_embeddings.shape[1])
index_flat.add(image_embeddings)
print(f'IndexFlatIP vectors: {index_flat.ntotal}')

## 5. Modern Retrieval Functions

Text-to-image and image-to-image are evaluated separately. A fused path is included because it builds a legitimate dual-query embedding by normalizing and averaging text and image query embeddings.

In [ ]:
def encode_text_query(query_text):
    inputs = processor(text=[query_text], return_tensors='pt', padding=True, truncation=True, max_length=77).to(device)
    with torch.no_grad():
        embedding = model.get_text_features(**inputs)
    if not torch.is_tensor(embedding):
        if hasattr(embedding, 'text_embeds') and embedding.text_embeds is not None:
            embedding = embedding.text_embeds
        elif hasattr(embedding, 'pooler_output') and embedding.pooler_output is not None:
            embedding = embedding.pooler_output
        elif hasattr(embedding, 'last_hidden_state'):
            embedding = embedding.last_hidden_state[:, 0, :]
        else:
            raise TypeError(f'Unsupported CLIP text output type: {type(embedding)}')
    embedding = embedding / embedding.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return embedding.cpu().numpy().astype('float32')


def encode_image_query(query_image):
    return encode_image_batch([query_image.convert('RGB')])


def search_index(query_embedding, exclude_image_id=None, top_k=10):
    search_k = min(top_k + (1 if exclude_image_id else 0), len(ordered_image_ids))
    scores, indices = index_flat.search(query_embedding.astype('float32'), search_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        image_id = ordered_image_ids[int(idx)]
        if exclude_image_id is not None and image_id == exclude_image_id:
            continue
        results.append({'image_id': image_id, 'score': float(score), 'rank': len(results) + 1})
        if len(results) >= top_k:
            break
    return results


def modern_text_to_image(query_text, exclude_image_id=None, top_k=10):
    return search_index(encode_text_query(query_text), exclude_image_id=exclude_image_id, top_k=top_k)


def modern_image_to_image(query_image, exclude_image_id=None, top_k=10):
    return search_index(encode_image_query(query_image), exclude_image_id=exclude_image_id, top_k=top_k)


def modern_fused_retrieval(query_text, query_image, text_weight=0.5, exclude_image_id=None, top_k=10):
    text_emb = encode_text_query(query_text)
    image_emb = encode_image_query(query_image)
    fused = text_weight * text_emb + (1.0 - text_weight) * image_emb
    fused = fused / np.maximum(np.linalg.norm(fused, axis=1, keepdims=True), 1e-12)
    return search_index(fused.astype('float32'), exclude_image_id=exclude_image_id, top_k=top_k)

## 6. Shared Evaluation Protocol

The same query file and caption-keyword relevance rules are used here so modern text-to-image can be compared with classic text-only, modern image-to-image with classic visual-only, and modern fused with classic fused.

In [ ]:
with QUERY_FILE.open('r', encoding='utf-8') as f:
    evaluation_queries = json.load(f)


def term_in_caption(term, caption):
    return normalize_text(term) in normalize_text(caption)


def captions_match_required(captions, required_terms):
    return all(any(term_in_caption(term, caption) for caption in captions) for term in required_terms)


def relevance_ids(query):
    relevant = set(query.get('manual_relevant_image_ids') or [])
    required_terms = query.get('required_terms') or []
    for _, row in df.iterrows():
        if captions_match_required(row['captions'], required_terms):
            relevant.add(row['image_id'])
    relevant.discard(query.get('reference_image_id'))
    return relevant


def resolve_reference_image(query, relevant):
    reference_id = query.get('reference_image_id')
    if reference_id in image_id_to_pos:
        return reference_id, full_ds[int(image_id_to_pos[reference_id])]['image']
    for candidate in sorted(relevant):
        if candidate in image_id_to_pos:
            warnings.warn(f'Reference image {reference_id} not found in this run; using {candidate} for {query["query_id"]}.')
            return candidate, full_ds[int(image_id_to_pos[candidate])]['image']
    warnings.warn(f'No available reference image for {query["query_id"]}; image/fused mode will be skipped.')
    return None, None


def queries_for_mode(mode):
    return [q for q in evaluation_queries if mode in q.get('query_modes', [])]


K_VALUES = [1, 5, 10, 20, 50]


def precision_at_k(retrieved_ids, relevant, k):
    return len(set(retrieved_ids[:k]) & relevant) / k if k else 0.0


def recall_at_k(retrieved_ids, relevant, k):
    return len(set(retrieved_ids[:k]) & relevant) / len(relevant) if relevant else 0.0


def summarize_metrics(per_query):
    summary = {}
    for k in K_VALUES:
        summary[str(k)] = {
            'avg_precision': float(np.mean([row[f'p@{k}'] for row in per_query])) if per_query else 0.0,
            'avg_recall': float(np.mean([row[f'r@{k}'] for row in per_query])) if per_query else 0.0,
        }
    return summary

In [ ]:
def evaluate_modern_text(text_queries, top_k=max(K_VALUES)):
    per_query = []
    for query in text_queries:
        relevant = relevance_ids(query)
        retrieved = modern_text_to_image(query['query_text'], exclude_image_id=query.get('reference_image_id'), top_k=top_k)
        retrieved_ids = [row['image_id'] for row in retrieved]
        row = {'query_id': query['query_id'], 'relevant_count': len(relevant), 'retrieved_ids': retrieved_ids}
        for k in K_VALUES:
            row[f'p@{k}'] = precision_at_k(retrieved_ids, relevant, k)
            row[f'r@{k}'] = recall_at_k(retrieved_ids, relevant, k)
        per_query.append(row)
    return {'per_query': per_query, 'summary': summarize_metrics(per_query)}


def evaluate_modern_image(image_queries, top_k=max(K_VALUES)):
    per_query = []
    for query in image_queries:
        relevant = relevance_ids(query)
        reference_id, query_image = resolve_reference_image(query, relevant)
        if query_image is None:
            continue
        retrieved = modern_image_to_image(query_image, exclude_image_id=reference_id, top_k=top_k)
        retrieved_ids = [row['image_id'] for row in retrieved]
        row = {'query_id': query['query_id'], 'reference_image_id': reference_id, 'relevant_count': len(relevant), 'retrieved_ids': retrieved_ids}
        for k in K_VALUES:
            row[f'p@{k}'] = precision_at_k(retrieved_ids, relevant, k)
            row[f'r@{k}'] = recall_at_k(retrieved_ids, relevant, k)
        per_query.append(row)
    return {'per_query': per_query, 'summary': summarize_metrics(per_query)}


def evaluate_modern_fused(dual_queries, text_weight=0.5, top_k=max(K_VALUES)):
    per_query = []
    for query in dual_queries:
        relevant = relevance_ids(query)
        reference_id, query_image = resolve_reference_image(query, relevant)
        if query_image is None:
            continue
        retrieved = modern_fused_retrieval(query['query_text'], query_image, text_weight=text_weight, exclude_image_id=reference_id, top_k=top_k)
        retrieved_ids = [row['image_id'] for row in retrieved]
        row = {'query_id': query['query_id'], 'reference_image_id': reference_id, 'relevant_count': len(relevant), 'retrieved_ids': retrieved_ids}
        for k in K_VALUES:
            row[f'p@{k}'] = precision_at_k(retrieved_ids, relevant, k)
            row[f'r@{k}'] = recall_at_k(retrieved_ids, relevant, k)
        per_query.append(row)
    return {'per_query': per_query, 'summary': summarize_metrics(per_query)}


modern_text = evaluate_modern_text(queries_for_mode('text'))
modern_image = evaluate_modern_image(queries_for_mode('image'))
modern_fused = evaluate_modern_fused(queries_for_mode('fused'))

modern_results = {
    'fast_dev': FAST_DEV,
    'model_name': MODEL_NAME,
    'faiss_index': 'IndexFlatIP',
    'k_values': K_VALUES,
    'text_to_image': modern_text,
    'image_to_image': modern_image,
    'fused': modern_fused,
}
print(json.dumps({mode: modern_results[mode]['summary'] for mode in ('text_to_image', 'image_to_image', 'fused')}, indent=2))

## 7. Classic vs Modern Comparison

This section overlays modern results with the saved Notebook 1 output when it is available.

In [ ]:
classic_path = OUTPUT_DIR / ('classic_results_fast_dev.json' if FAST_DEV else 'classic_results_full.json')
classic_results = json.loads(classic_path.read_text(encoding='utf-8')) if classic_path.exists() else None
if classic_results is None:
    print(f'Classic results not found at {classic_path}; run Notebook 1 first for direct overlay.')
else:
    print(f'Loaded classic comparison data from {classic_path}')


def plot_comparison(classic_results, modern_results):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    series = []
    if classic_results is not None:
        series.extend([
            ('classic text-only', classic_results['text_only']['summary']),
            ('classic visual-only', classic_results['visual_only']['summary']),
            ('classic fused', classic_results['fused']['summary']),
        ])
    series.extend([
        ('modern text-to-image', modern_results['text_to_image']['summary']),
        ('modern image-to-image', modern_results['image_to_image']['summary']),
        ('modern fused', modern_results['fused']['summary']),
    ])
    for label, summary in series:
        axes[0].plot(K_VALUES, [summary[str(k)]['avg_precision'] for k in K_VALUES], marker='o', label=label)
        axes[1].plot(K_VALUES, [summary[str(k)]['avg_recall'] for k in K_VALUES], marker='o', label=label)
    axes[0].set_title('Average Precision@K')
    axes[1].set_title('Average Recall@K')
    for ax in axes:
        ax.set_xlabel('K')
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)
    fig.tight_layout()
    return fig


fig = plot_comparison(classic_results, modern_results)
if not FAST_DEV:
    fig.savefig(FIGURE_DIR / 'classic_modern_clip_comparison.png', dpi=150)
plt.show()

## 8. Optional Scalability / Latency Section

`IndexFlatIP` remains the main Flickr8k result. The IVF/nprobe experiment below is optional and only illustrates the latency/recall trade-off for larger collections.

In [ ]:
if len(image_embeddings) >= 100:
    nlist = min(50, max(2, int(np.sqrt(len(image_embeddings)))))
    quantizer = faiss.IndexFlatIP(image_embeddings.shape[1])
    index_ivf = faiss.IndexIVFFlat(quantizer, image_embeddings.shape[1], nlist, faiss.METRIC_INNER_PRODUCT)
    index_ivf.train(image_embeddings)
    index_ivf.add(image_embeddings)
    nprobe_values = sorted(set([1, min(2, nlist), min(5, nlist), min(10, nlist)]))
    latency_rows = []
    sample_query = image_embeddings[0:1]
    for nprobe in nprobe_values:
        index_ivf.nprobe = nprobe
        start = time.time()
        for _ in range(20):
            index_ivf.search(sample_query, 10)
        latency_rows.append({'nprobe': int(nprobe), 'latency_ms': (time.time() - start) / 20 * 1000})
    print(pd.DataFrame(latency_rows))
else:
    print('Skipping optional IVF demo: not enough images in this run.')

## 9. Save Results

FAST_DEV outputs are debugging artifacts. Full-run outputs are the ones to inspect before creating any final slide narrative.

In [ ]:
result_path = OUTPUT_DIR / ('modern_clip_results_fast_dev.json' if FAST_DEV else 'modern_clip_results_full.json')
result_path.write_text(json.dumps(modern_results, indent=2), encoding='utf-8')
print(f'Saved results to {result_path}')